# 2. PIM and Conditional Access - Implementation

SC-900 taught you *what* Privileged Identity Management (PIM) and Conditional Access (CA) are.
AZ-500 expects you to *configure* them.

## Before you run this notebook

1. Run `uv sync` in the lab folder.
2. In VS Code, pick the `.venv` kernel in the top-right kernel picker.
3. Reload the window (`Cmd+Shift+P` then *Reload Window*) if the kernel doesn't appear.

No Docker required - everything runs as plain Python.

## PIM - why standing access is dangerous

Most compromises we read about begin with **standing privileged access**: a Global Admin account that's always on, always valid, always a juicy target. PIM flips this: admins are **eligible** for a role and must **activate** it for a short time, with MFA and a justification.

### Bad to Best


In [ ]:
scenarios = [
    ('BAD',  'Alice is a permanent Global Administrator.',
     '1 compromised password => full tenant takeover, and nobody notices until audit.'),
    ('OKAY', 'Alice is Global Administrator, but her account is MFA-enforced.',
     'Much better, but her credentials are valuable 24/7 and phishing can still bypass MFA.'),
    ('BEST', 'Alice is PIM-eligible for Global Administrator, max 2h activation, MFA + approval + justification.',
     'Most of the time she is a regular user. Attackers steal nothing valuable unless she is actively activating.'),
]
for label, setup, impact in scenarios:
    print(f'{label:<5}  {setup}')
    print(f'        -> {impact}\n')


### PIM settings you must configure

For each role, PIM has per-role settings:

| Setting | Options | Typical default |
|---------|---------|-----------------|
| **Maximum activation duration** | 0.5 to 24 hours | 8 hours |
| **Require MFA on activation** | Yes / No | Yes for privileged roles |
| **Require justification** | Yes / No | Yes |
| **Require approval** | Yes / No + who approves | Yes for Global Admin |
| **Require ticket info** | Yes / No | No |
| **Allow permanent eligible** | Yes / No | No (set expiration) |
| **Allow permanent active** | Yes / No | No |
| **Notification on activation** | Email to admins | Yes |


In [ ]:
import json
from datetime import datetime, timedelta

PIM_ROLE_SETTINGS = {
    'Global Administrator': {
        'max_activation_hours': 2,
        'require_mfa': True,
        'require_justification': True,
        'require_approval': True,
        'approvers': ['security-team@contoso.com'],
        'eligible_max_months': 6,
        'permanent_eligible': False,
    },
    'Contributor': {
        'max_activation_hours': 8,
        'require_mfa': True,
        'require_justification': True,
        'require_approval': False,
        'approvers': [],
        'eligible_max_months': 12,
        'permanent_eligible': False,
    },
    'Security Reader': {
        'max_activation_hours': 8,
        'require_mfa': False,
        'require_justification': False,
        'require_approval': False,
        'approvers': [],
        'eligible_max_months': 12,
        'permanent_eligible': True,
    },
}

def activate_pim_role(user, role, justification, mfa, ticket=''):
    s = PIM_ROLE_SETTINGS.get(role)
    if not s:
        return {'status': 'error', 'reason': f'Role {role} not found in PIM'}

    failed = []
    if s['require_mfa'] and not mfa:
        failed.append('MFA required but not presented')
    if s['require_justification'] and not justification:
        failed.append('Justification required')

    now = datetime.now()
    if failed:
        return {'status': 'DENIED', 'failed_checks': failed}
    if s['require_approval']:
        return {
            'status': 'PENDING APPROVAL',
            'role': role, 'user': user,
            'justification': justification,
            'approvers': s['approvers'],
            'would_expire': (now + timedelta(hours=s['max_activation_hours'])).strftime('%Y-%m-%d %H:%M'),
        }
    return {
        'status': 'ACTIVATED',
        'role': role, 'user': user,
        'active_until': (now + timedelta(hours=s['max_activation_hours'])).strftime('%Y-%m-%d %H:%M'),
    }

print('=== PIM activation scenarios ===\n')
print('--- 1. Global Admin (requires approval) ---')
print(json.dumps(activate_pim_role('alice', 'Global Administrator', 'Emergency CA policy fix', True), indent=2))

print('\n--- 2. Contributor (no approval, just MFA + justification) ---')
print(json.dumps(activate_pim_role('bob', 'Contributor', 'Deploy hotfix to prod', True), indent=2))

print('\n--- 3. Contributor without MFA (blocked) ---')
print(json.dumps(activate_pim_role('bob', 'Contributor', 'Deploy hotfix', False), indent=2))

print('\n--- 4. Global Admin missing justification (blocked) ---')
print(json.dumps(activate_pim_role('alice', 'Global Administrator', '', True), indent=2))


### Activating a PIM role via the Microsoft Graph API

```bash
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/roleManagement/directory/roleAssignmentScheduleRequests' \
  --body '{
    "action": "selfActivate",
    "justification": "Emergency CA policy fix",
    "roleDefinitionId": "<role-id>",
    "directoryScopeId": "/",
    "principalId": "<user-id>",
    "scheduleInfo": {
      "startDateTime": "2026-04-17T10:00:00Z",
      "expiration": {"type": "afterDuration", "duration": "PT2H"}
    }
  }'
```

---
## Conditional Access - policy structure

Every CA policy has three parts:

1. **Assignments** - *who* (users/groups), *what* (apps), *where* (conditions like IP, device, risk).
2. **Access controls** - grant / block, plus requirements (MFA, compliant device, approved app).
3. **Session controls** - sign-in frequency, persistent browser, app-enforced restrictions.


In [ ]:
CA_POLICIES = [
    {
        'name': 'Require MFA for admins',
        'state': 'enabled',
        'assignments': {
            'users': {'include_roles': ['Global Administrator', 'Security Administrator', 'Exchange Administrator'],
                      'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
        },
        'grant': {'operator': 'AND', 'controls': ['mfa']},
        'scenario': 'Baseline: admins always need MFA. Break-glass accounts are excluded.',
    },
    {
        'name': 'Block legacy authentication',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
            'conditions': {'client_apps': ['exchangeActiveSync', 'other']},
        },
        'grant': {'operator': 'OR', 'controls': ['block']},
        'scenario': 'Legacy protocols (IMAP/POP3/SMTP) cannot do MFA. Block them.',
    },
    {
        'name': 'Require compliant device for Azure portal',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': ['Microsoft Azure Management']},
        },
        'grant': {'operator': 'AND', 'controls': ['compliant_device']},
        'scenario': 'Only Intune-managed devices can reach the Azure portal.',
    },
    {
        'name': 'Require MFA + password reset for risky sign-ins',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
            'conditions': {'sign_in_risk': ['medium', 'high']},
        },
        'grant': {'operator': 'AND', 'controls': ['mfa', 'password_change']},
        'scenario': 'Entra ID Protection flags risky sign-ins. Force MFA + password reset.',
    },
    {
        'name': 'Block access from untrusted countries',
        'state': 'report-only',
        'assignments': {
            'users': {'include': 'all', 'exclude': ['group:break-glass-accounts']},
            'apps':  {'include': 'all'},
            'conditions': {'locations': {'include': 'all', 'exclude': ['named:TrustedCountries']}},
        },
        'grant': {'operator': 'OR', 'controls': ['block']},
        'scenario': 'Start in report-only, review sign-in logs, then enable.',
    },
]

print('=== Conditional Access policies ===\n')
for p in CA_POLICIES:
    state_tag = {'enabled': '[ON]', 'report-only': '[REPORT]', 'disabled': '[OFF]'}[p['state']]
    print(f'{state_tag:<10} {p["name"]}')
    print(f'   Scenario: {p["scenario"]}')
    print(f'   Grant:    {p["grant"]["operator"]} -> {p["grant"]["controls"]}\n')


## Break-glass accounts - the escape hatch

If Conditional Access breaks (misconfigured policy, MFA provider outage, expired certificate), you can lock *everyone* out, including yourself. To prevent that, create **two emergency access accounts** ("break-glass"):

| Setting | Value |
|---------|-------|
| Count | 2, minimum |
| Type | Cloud-only accounts, not synced from AD |
| Username | Not using domain names someone would guess |
| MFA | FIDO2 security key stored in a safe - **not** phone-based |
| CA policies | **Excluded from all of them** (including MFA policies!) |
| Monitoring | Alert on any sign-in using these accounts |

Let's simulate what happens when you forget to exclude them.


In [ ]:
# The MFA provider is down: nobody can satisfy MFA.
user_has_mfa = {'alice': False, 'break-glass-1': False}

def user_allowed(user, policies):
    reasons = []
    for p in policies:
        if p['state'] != 'enabled':
            continue
        excluded = user in p['assignments']['users'].get('exclude', [])
        controls = p['grant']['controls']
        if 'block' in controls and not excluded:
            reasons.append(f'BLOCKED by "{p["name"]}"')
        elif 'mfa' in controls and not excluded and not user_has_mfa.get(user):
            reasons.append(f'MFA required by "{p["name"]}" and not satisfied')
    return reasons

# Scenario A: break-glass is NOT excluded.
bad_policies = [{
    'name': 'Require MFA for admins',
    'state': 'enabled',
    'assignments': {'users': {'include': 'all', 'exclude': []}, 'apps': {'include': 'all'}},
    'grant': {'operator': 'AND', 'controls': ['mfa']},
}]
print('--- MFA provider is DOWN ---')
print('Policy misconfigured (break-glass NOT excluded):')
print(f'  alice          -> {user_allowed("alice", bad_policies) or ["allowed"]}')
print(f'  break-glass-1  -> {user_allowed("break-glass-1", bad_policies) or ["allowed"]}')
print('   !! Nobody can sign in - the tenant is locked. !!\n')

# Scenario B: break-glass IS excluded.
good_policies = [dict(
    bad_policies[0],
    assignments={'users': {'include': 'all', 'exclude': ['break-glass-1']}, 'apps': {'include': 'all'}},
)]
print('Policy correct (break-glass excluded):')
print(f'  alice          -> {user_allowed("alice", good_policies) or ["allowed"]}')
print(f'  break-glass-1  -> {user_allowed("break-glass-1", good_policies) or ["allowed"]}')
print('   OK - ops can sign in with break-glass and fix the broken policy.')


## Named locations

Location-based policies need a **named location** defined first:

```bash
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/identity/conditionalAccess/namedLocations' \
  --body '{
    "@odata.type": "#microsoft.graph.ipNamedLocation",
    "displayName": "Corporate offices",
    "isTrusted": true,
    "ipRanges": [
      {"@odata.type": "#microsoft.graph.iPv4CidrRange", "cidrAddress": "203.0.113.0/24"},
      {"@odata.type": "#microsoft.graph.iPv4CidrRange", "cidrAddress": "198.51.100.0/24"}
    ]
  }'
```

## Entra ID Protection - risk signals

Conditional Access policies can react to **risk** signals from Entra ID Protection:

| Risk type | Examples of signals |
|-----------|---------------------|
| **User risk** | Leaked credentials (password on the dark web), anomalous user behaviour |
| **Sign-in risk** | Impossible travel, unfamiliar sign-in properties, anonymous IP (Tor), malware-linked IP |

Typical policy combos:

| If... | Then... |
|-------|---------|
| User risk = *high* | Force **password reset** (blocks until done) |
| Sign-in risk = *medium/high* | Force **MFA** |

## Exam tips

- CA policies are **additive** - the most restrictive wins.
- CA **cannot grant** access that isn't already permitted by RBAC.
- Always start a new policy in **report-only** mode. Review sign-in logs, then enable.
- **Break-glass** accounts: excluded from *all* CA policies, alert on every sign-in.
- Requires Entra ID **Premium P1** (CA) or **P2** (PIM, Identity Protection).

---
## Summary

| Implementation | Key details |
|---------------|-------------|
| **PIM** | Eligible -> activate with MFA + justification + (optional) approval |
| **CA structure** | Assignments -> Grant controls -> Session controls |
| **Named locations** | Define trusted IPs before using location-based policies |
| **Report-only mode** | Always test before enforcing |
| **Break-glass accounts** | Exclude from *all* CA policies; alert on any sign-in |
| **Identity Protection** | Feeds user-risk and sign-in-risk signals into CA |

**Next**: [Notebook 3 - App registrations and managed identities](03_app_registrations_and_managed_identities.ipynb)
